<span style="color:red; font-weight:bold" >Module3</span>

In this final model we're gonna push our agent beyond the prototype stage.      
We'll start by introducing middleware    
What are the most important concepts in modern agent design ?   

Middleware lets you intercept and customize your agent's execution at every step.      
We'll learn how to dynamically swap in tools, adjust prompts on the fly, and even change the underlying model based on the situation your agent finds itself in.      
We'll tackle long conversation management. Context windows are finite, but great conversation shouldn't be.     
We'll learn how to summarize, compress, and intelligently retain information so your agent can stay coherent over hours or days of interaction without forgetting what matters.     

Next we'll look at human in the loop patterns.   
Not every action shoul be left entirely up to an AI system, especially sensitive ones.     
We'll learn how to insert approval checkpoints. Where your agent and its human counterpart work together seamlessly.    

Up until now our agents have been powerful but they've also been fairly static.    
Fixed tools, fixed prompts, fixed models,, Real world AI apps don't work like that! and neither will ours after this Module 😊 .

<span style="font-weight:bold">==> By the end of this Module, we'll put all of this together to build an email assistant capable of automating an entier inbox !! Safely , Reliably and with all the custom logic we've just designed !!</span>

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

<span style="color:rgb(200, 100, 51); font-weight:bold"> What is Middleware and why do we need it ? </span>

Middleware is really a catch all term, we use for functions that we can insert within these loops, which allow us to control and customize our agent's executions


Take for example, a customer facing payroll assistant that we want to make sure isn't providing legal advice to users.     
==> Well,, we can just build a classifier, Legal/not Legal, and insert it as custom middleware after the agent has run. 

Or maybe we've got a customer support agent that can process refunds.   
It might not be something that we want to be doing without human oversight.    
Well, we can just include our human in the loop middleware function before that tool is called everytime.


<span style="font-weight:bold"> ==> So basically, middleware is the key to leveling up your hobby project to an agent ready for real users.</span>

With help from our checkpointer, we've got an agent that maintains a list of messages so it can remember what happened in our conversation to date.

This works well for shorter conversations, but after a while, the list becomes longer and longer, so it overflows our agent's context window.          

It literally can't parse through all this information in an efficient way, slowing down our app, impacting performance, and increasing its cost. 

There is two ways to fix this problem: Summarizing the conversation and trimming or deleting messages.    



In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Get your key from environment
google_key = os.getenv("GOOGLE_API_KEY")
print(f"Key exists: {bool(google_key)}")

# Create the model
model = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",  # Use flash for free tier
    google_api_key=google_key
)

Key exists: True


In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model= model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model= model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [6]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\nThe capital of the moon is Lunapolis. Lunapolis has clear skies, with a high of 120C and a low of -100C. There are 100,000 cheese miners living in Lunapolis, and their union is expected to strike due to unhappiness with the new president.', additional_kwargs={}, response_metadata={}, id='c95ac7b1-aa9d-497e-98e3-bf4da5894913'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='bbf704b4-7ff2-4c8d-b968-260e0210f0d2'),
              AIMessage(content='As Lunapolis\' new president, my immediate priority would be to prevent the strike, or at the very least, de-escalate the situation and open a productive dialogue. The well-being of 100,000 cheese miners is not just a labor issue; it\'s the economic backbone and social fabric of Lunapolis.\n\nHere\'s how I would respond:\n\n1.  **Immedi

In [7]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

The capital of the moon is Lunapolis. Lunapolis has clear skies, with a high of 120C and a low of -100C. There are 100,000 cheese miners living in Lunapolis, and their union is expected to strike due to unhappiness with the new president.


## Trim/delete messages

In [8]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [9]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [10]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='dba7f109-c485-4941-acdd-f072c2f95179'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='b06ed76b-33d1-469c-930e-183de25482fb'),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='c5a695e9-63e1-4fd6-ac6f-eacb7732e93d'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='b720d110-2af9-41fb-a3a1-cde51227d61f'),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='9702b454-0e4c-456a-8b95-85dd116ef606'),
              AIMessage(content="As an AI, I don't have access to your physical device, so I can't tell you its temperature.\n\nHowever, the device's temperature can be a 

In [11]:
print(response["messages"][-1].content)

As an AI, I don't have access to your physical device, so I can't tell you its temperature.

However, the device's temperature can be a clue. Is the device unusually hot or cold to the touch?

And circling back to my previous question, **is the device showing any lights or indicators at all?** (Like a power light, charging light, or any blinking lights?)


In [12]:
print(response["messages"][-2].content)

What's the temperature of the device?


In [13]:
print(response["messages"][-3].content)

Is the device showing any lights or indicators?
